# معمل بنية الرسائل والبيانات الوصفية وتتبع التوكنز (`Messages`)

أهلاً بك في المعمل العملي للمحاضرة 62!

في المعمل ده هنتعرف على أنواع الرسائل الأربعة الأساسية في `LangChain` وإزاي بتتحكم في سلوك الموديل:
1. الفرق بين السؤال النصي المباشر (`Text Prompt`) وقائمة الرسائل (`Message Prompt`).
2. قوة رسالة النظام (`SystemMessage`) لما تكتبها بتفصيل واحترافية.
3. إضافة `Metadata` على رسالة المستخدم + كتابة `AIMessage` يدوية + قراءة استهلاك التوكنز `usage_metadata`.
4. ربط رسالة الأداة (`ToolMessage`) برقم الطلب!


In [1]:
# تجهيز الموديل واستيراد أنواع الرسائل
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

load_dotenv()
model = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
print("تم تجهيز الموديل وأنواع الرسائل بنجاح.")


تم تجهيز الموديل وأنواع الرسائل بنجاح.


### 1. تأثير قوة `SystemMessage` على إجابة الموديل

لاحظ الفرق: لو سألت الموديل "كيف أنشئ REST API؟" برسالة نظام عامة، ممكن يجاوب بأي لغة. لكن لما نحدد في `SystemMessage` إنه مبرمج بايثون خبير بـ FastAPI، الإجابة بتكون مركزة وعملية 100%!


In [2]:
messages = [
    SystemMessage(content="You are a senior Python developer expert in FastAPI. Answer concisely in plain text with 1 short code snippet and no emojis."),
    HumanMessage(content="How do I create a simple REST API endpoint?")
]

res = model.invoke(messages)
print("رد الموديل الموجه بالـ SystemMessage:")
print(res.content[:400])


رد الموديل الموجه بالـ SystemMessage:
Create a FastAPI app, define a route, and run it with Uvicorn.

```python
# main.py
from fastapi import FastAPI

app = FastAPI()

@app.get("/hello")
def read_hello():
    return {"message": "Hello, world!"}
```

Run with:

```bash
uvicorn main:app --reload
```

The endpoint `/hello` will respond with `{"message":"Hello, world!"}`.


### 2. تتبع هوية المستخدم وقراءة تكلفة التوكنز (`usage_metadata`)

تقدر تضيف اسم المستخدم (`name`) ورقم تعريفي (`id`) على `HumanMessage`، وكمان تقرا من رد الموديل استهلاك التوكنز بالتفصيل!


In [3]:
history = [
    SystemMessage(content="Be very concise and friendly. Plain text only without emojis."),
    HumanMessage(content="Can you help me?", name="Alice", id="msg_001"),
    AIMessage(content="Yes! I would be happy to help you."),
    HumanMessage(content="What is 15 + 27?", name="Alice", id="msg_002")
]

reply = model.invoke(history)
print("رد الموديل:", reply.content)
print("تفاصيل التوكنز المستهلكة (usage_metadata):")
print(reply.usage_metadata)


رد الموديل: 42
تفاصيل التوكنز المستهلكة (usage_metadata):
{'input_tokens': 119, 'output_tokens': 29, 'total_tokens': 148, 'output_token_details': {'reasoning': 19}}


### 3. تجربة رسالة الأداة (`ToolMessage`)

لما الموديل يطلب أداة برقم `id` معين، إحنا بنرد عليه بـ `ToolMessage` شايلة نتيجة الأداة ونفس الـ `tool_call_id`. تعال نجرب نبنيها بيدينا!


In [4]:
tool_history = [
    HumanMessage(content="What is the weather in Alexandria?"),
    AIMessage(content="", tool_calls=[{"name": "get_weather", "args": {"city": "Alexandria"}, "id": "call_alex_01"}]),
    ToolMessage(content="Sunny, 24C, gentle sea breeze.", tool_call_id="call_alex_01")
]

final_weather = model.invoke(tool_history)
print("الرد النهائي بناءً على الـ ToolMessage:")
print(final_weather.content)


الرد النهائي بناءً على الـ ToolMessage:
**Alexandria Weather**  
☀️ Sunny, 24 °C with a gentle sea breeze.


### 4. جرب بنفسك!

دلوقتي دورك في التجربة الحرة:
1. جرب تغير شخصية الموديل في `SystemMessage` (مثلاً: خليه "مدرس تاريخ بيشرح بأسلوب قصصي مشوق" أو "مهندس أمن سيبراني صارم").
2. لاحظ إزاي عدد توكنز الـ `input_tokens` و `output_tokens` بيتغير في `usage_metadata`!


In [5]:
# جرب بنفسك هنا: غيّر الشخصية وشوف النتيجة والتوكنز
my_msgs = [
    SystemMessage(content="أنت مهندس برمجيات خبير تشرح المصطلحات ببساطة في سطرين بالعامية المصرية بدون إيموجي."),
    HumanMessage(content="إيه فايدة الـ SystemMessage في الشات بوت؟", name="Developer_Ali", id="dev_99")
]

my_reply = model.invoke(my_msgs)
print("الرد:", my_reply.content)
print("إجمالي التوكنز:", my_reply.usage_metadata["total_tokens"])


الرد: الـ SystemMessage هو الرسالة اللي بيحطها المطور قبل أي تفاعل، وبتحدد قواعد اللعبة للـ AI زي الأسلوب أو الموضوع. بيساعد على إن النموذج يشتغل في نطاق محدد ويطلع ردود متناسقة مع المطلوب.
إجمالي التوكنز: 253
